1. Importing Necessary Libraries

In [1]:
import os, sys, math, json, csv, random, itertools, time
import numpy as np
import pandas as pd
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers as L, models as KM
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import AUC
from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt

print("TF:", tf.__version__)
SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()


TF: 2.15.1


2. Configuration

In [2]:
# Paths & core hyperparameters
CSV_PATH = '../../annotations/train_detector_annotations.csv'  # merged annotations file (clean + adversarial)
IMG_SIZE = 50                                # must match your detector's expected input size
BATCH_L = 64                                 # labeled batch size
BATCH_U = 256                                # unlabeled batch size (often ~4x labeled)
EPOCHS  = 30
LR      = 3e-4

# FixMatch-specific parameters
TAU = 0.95          # confidence threshold for pseudo-labeling
LAMBDA_U = 1.5      # weight for unsupervised loss term
EMA_ALPHA = 0.999   # default EMA decay for model averaging

# Optional: on-the-fly class rebalancing for labeled data
# Example: CLASS_WEIGHT = {0: 1.0, 1: 1.5}
CLASS_WEIGHT = None  

# Scheduled configs (progressive thresholding, ramp-up, EMA warm/cool)
TAU_START, TAU_MID, TAU_FINAL = 0.80, 0.90, 0.95       # staged pseudo-label thresholds 
LAMBDA_U_MAX = 1.5                                     # target max for unsupervised weight
WARMUP_EPOCHS = 2                                      # epochs with zero unsupervised loss
EMA_ALPHA_WARM = 0.90                                  # faster-tracking EMA early on
EMA_ALPHA_LATE = 0.999                                 # more stable EMA later


def get_tau(ep):
    """Return the pseudo-label confidence threshold for epoch `ep`.
    Uses coarse staging: start -> mid -> final."""
    if ep < 5:    return TAU_START
    elif ep < 10: return TAU_MID
    else:         return TAU_FINAL

def get_lambda_u(ep):
    """Linear ramp-up of the unsupervised loss weight after a warmup period.
    - 0 until WARMUP_EPOCHS
    - linear increase to LAMBDA_U_MAX by epoch 10
    """
    if ep < WARMUP_EPOCHS: return 0.0
    # linear ramp until epoch 10 (inclusive of cap)
    t = min(1.0, (ep - WARMUP_EPOCHS) / (10 - WARMUP_EPOCHS))
    return t * LAMBDA_U_MAX

def get_ema_alpha(ep):
    """EMA decay schedule: faster adaptation early, more stability later."""
    return EMA_ALPHA_WARM if ep < 3 else EMA_ALPHA_LATE

3. Load and Validate Annotations CSV, Enforce Dtypes, and Quick Sanity Checks

In [3]:
# Load CSV
# Ensure the annotations file exists before reading
assert os.path.exists(CSV_PATH), f"CSV not found: {CSV_PATH}"

# Read the merged annotations (clean + adversarial)
df = pd.read_csv(CSV_PATH)

df["image_id"] = (
    df["image_id"]
    .astype(str)
    .str.strip()
    .str.replace("\\", "/", regex=False)
    .apply(lambda p: p if p.startswith("../../datasets/") else "../../datasets/" + p)
)

# Verify the schema is what we expect
expected_cols = {"image_id","adv_label","split","isNight"}
missing = expected_cols - set(df.columns)
assert not missing, f"CSV missing columns: {missing}"

# Type casts for consistent downstream processing
df["adv_label"] = df["adv_label"].astype(int)   # binary target for adversarial presence
df["isNight"]   = df["isNight"].astype(int)     # binary flag for night-time images
df["split"]     = df["split"].astype(str)       # dataset split identifier (e.g., train/val/test)

# Quick peek at the data and split distribution for sanity
print(df.head())
print(df['split'].value_counts())

                                          image_id  label  isNight  split  \
0  ../../datasets/processed_images/train/img_0.jpg      1        0  train   
1  ../../datasets/processed_images/train/img_1.jpg      1        0  train   
2  ../../datasets/processed_images/train/img_2.jpg      1        0  train   
3  ../../datasets/processed_images/train/img_3.jpg      1        0  train   
4  ../../datasets/processed_images/train/img_4.jpg      1        0  train   

   adv_label  
0          0  
1          0  
2          0  
3          0  
4          0  
split
train    93930
test     34797
val      26751
Name: count, dtype: int64


4. Create Labeled Train/Val/Test Splits and Build Unlabeled Pool

In [4]:

# Split the dataframe into train/val/test subsets based on the `split` column
df_train = df[df["split"]=="train"].copy()
df_val   = df[df["split"]=="val"].copy()
df_test  = df[df["split"]=="test"].copy()

# Labeled subsets: we will use labels only for train/val (and test for final evaluation)
train_labeled = df_train.copy()
val_labeled   = df_val.copy()
test_labeled  = df_test.copy()

# Unlabeled pool: can be sourced from train + val without using labels (for semi-supervised learning)
unlabeled_pool = pd.concat([df_train, df_val], axis=0).reset_index(drop=True)

# Quick counts for sanity checking
print("Labeled train:", len(train_labeled), 
      "Val:", len(val_labeled), 
      "Test:", len(test_labeled),
      "Unlabeled:", len(unlabeled_pool))

Labeled train: 93930 Val: 26751 Test: 34797 Unlabeled: 120681


5. TensorFlow Data Utilities: Image Loader and Weak/Strong Augmentations

In [5]:
# TF data utils
AUTOTUNE = tf.data.AUTOTUNE

def fix_path_tf(path):
    path = tf.strings.regex_replace(path, r"\\", "/")
    has_prefix = tf.strings.regex_full_match(path, r"\.\./\.\./datasets/.*")
    path = tf.cond(
        has_prefix,
        lambda: path,
        lambda: tf.strings.join(["../../datasets/", path])
    )
    return path

def load_image(path):
    tf.print("PATH BEFORE:", path)

    path = fix_path_tf(path)

    tf.print("PATH AFTER:", path)

    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img
def weak_aug(img):
    """Light augmentation used for pseudo-label generation (keeps semantics stable)."""
    img = tf.image.random_flip_left_right(img)
    return img

def strong_aug(img):
    """Stronger augmentation for consistency regularization (FixMatch-style)."""
    # Spatial symmetry
    img = tf.image.random_flip_left_right(img)

    # Color jitter
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    img = tf.image.random_hue(img, 0.05)

    # Simple Cutout: zero-out a random square patch
    size = tf.random.uniform([], 8, 16, dtype=tf.int32)                 # patch side length
    x0 = tf.random.uniform([], 0, IMG_SIZE - size, dtype=tf.int32)      # top-left x
    y0 = tf.random.uniform([], 0, IMG_SIZE - size, dtype=tf.int32)      # top-left y


    mask = tf.ones((size, size, 3), dtype=img.dtype)                            # ones => to be zeroed later
    paddings = [[y0, IMG_SIZE - y0 - size], [x0, IMG_SIZE - x0 - size], [0,0]]
    patch = tf.pad(mask, paddings, constant_values=0.0)                         # place the square at (x0, y0)

    # Where patch==1, set image to 0 (black), else keep original pixel
    img = tf.where(patch==1.0, 0.0, img)
    return img

def to_float32(x):
    """Utility to ensure tensors are float32 (useful in tf.data pipelines)."""
    return tf.cast(x, tf.float32)

6. Labeled tf.data Pipeline: (Image + Metadata) → Target, with Shuffling/Repeat/Prefetch

In [6]:
# Labeled dataset (image + meta -> y)
def build_labeled_ds(frame, shuffle=True, repeat=False, batch=BATCH_L):
    """
    Build a labeled tf.data.Dataset that yields:
        ((image, meta), y)
    where:
        - image: float32 tensor in [0,1], shape [IMG_SIZE, IMG_SIZE, 3]
        - meta:  float32 tensor with the `isNight` flag, shape [1]
        - y:     float32 tensor with the binary target `adv_label`, shape [1]
    """
    # Extract columns from the dataframe
    paths = frame["image_id"].values
    ys    = frame["adv_label"].values.astype("float32")
    ms    = frame["isNight"].values.astype("float32")

    # Create a dataset of (path, y, m) tuples
    ds = tf.data.Dataset.from_tensor_slices((paths, ys, ms))

    # Optional randomization for better generalization
    if shuffle: ds = ds.shuffle(buffer_size=len(frame), seed=SEED, reshuffle_each_iteration=True)

    def _map(path, y, m):
        # Load & lightly augment image (keep semantics stable for labeled data)
        img = load_image(path)
        img = weak_aug(img)  
        
        # Ensure scalar shapes [1] for metadata and label
        m   = tf.reshape(to_float32(m), [1])
        y   = tf.reshape(to_float32(y), [1])

        # Return ((image, meta), label) to match a model with two inputs
        return (img, m), y

    # Map with parallel calls for throughput
    ds = ds.map(_map, num_parallel_calls=AUTOTUNE)

    # Optionally repeat indefinitely (useful for training)
    if repeat: ds = ds.repeat()

    # Batch and prefetch to overlap producer/consumer
    ds = ds.batch(batch).prefetch(AUTOTUNE)
    return ds

# Build datasets for train/val/test (labeled only)
ds_train_l = build_labeled_ds(train_labeled, shuffle=True, repeat=True, batch=BATCH_L)
ds_val_l   = build_labeled_ds(val_labeled, shuffle=False, repeat=False, batch=BATCH_L)
ds_test_l  = build_labeled_ds(test_labeled, shuffle=False, repeat=False, batch=BATCH_L)

# Steps are computed with floor division; guard with max(1, ...) to avoid zero
ds_steps_per_epoch = max(1, len(train_labeled)//BATCH_L)
val_steps = max(1, len(val_labeled)//BATCH_L)
test_steps= max(1, len(test_labeled)//BATCH_L)

print("steps/epoch:", ds_steps_per_epoch, "val:", val_steps, "test:", test_steps)

steps/epoch: 1467 val: 417 test: 543


7. Unlabeled tf.data Pipeline: Weak/Strong Views + Metadata (No Labels Used)

In [7]:
# Unlabeled dataset (weak/strong + meta only)
def build_unlabeled_ds(frame, batch=BATCH_U):
    """
    Build an unlabeled tf.data.Dataset that yields:
        (weak_view, strong_view, meta)
    where:
        - weak_view:  lightly augmented image (for pseudo-label prediction)
        - strong_view: strongly augmented image (for consistency training)
        - meta:       float32 tensor with the `isNight` flag, shape [1]
    Notes:
        * Labels from `frame` are intentionally ignored (semi-supervised setting).
        * Dataset is repeated indefinitely to stay in sync with the labeled stream.
    """
    paths = frame["image_id"].values
    ms    = frame["isNight"].values.astype("float32")

    # Shuffle to ensure diverse unlabeled batches
    ds = tf.data.Dataset.from_tensor_slices((paths, ms)).shuffle(len(frame), seed=SEED)

    def _map(path, m):
        # Load once; branch into weak/strong augmented views
        img = load_image(path)
        w   = weak_aug(img)                     # used to compute pseudo-labels
        s   = strong_aug(tf.identity(img))      # stronger distortions for consistency loss

        # Ensure meta is a rank-1 tensor (shape [1])
        m   = tf.reshape(to_float32(m), [1])
        return (w, s, m)
    
    ds = ds.map(_map, num_parallel_calls=AUTOTUNE)
    ds = ds.repeat()  # infinite stream to match supervised steps per epoch
    ds = ds.batch(batch).prefetch(AUTOTUNE)
    return ds

# Build the unlabeled dataset (used only for semi-supervised training)
ds_unlabeled = build_unlabeled_ds(unlabeled_pool, batch=BATCH_U)

8. CNN Binary Detector with Image + Metadata Inputs (Student/Teacher Models)

In [8]:
# CNN detector with image + meta (isNight)
def build_cnn_binary(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    """
    Build a lightweight CNN for binary classification that consumes:
      - image input: RGB tensor of shape (IMG_SIZE, IMG_SIZE, 3)
      - meta input:  scalar feature `isNight` with shape (1,)
    Returns a Keras Model with a sigmoid output (probability of positive class).
    """
    # Image input branch
    img_in  = L.Input(shape=input_shape, name="image")
    meta_in = L.Input(shape=(1,), name="meta_isNight")

    # Feature extractor (simple ConvNet backbone)
    x = L.Conv2D(32, 3, padding="same", activation="relu")(img_in)
    x = L.BatchNormalization()(x)
    x = L.MaxPool2D()(x)

    x = L.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = L.BatchNormalization()(x)
    x = L.MaxPool2D()(x)

    x = L.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = L.BatchNormalization()(x)
    x = L.GlobalAveragePooling2D()(x)

    # Concatenate image features with metadata (isNight)
    x = L.Concatenate()([x, meta_in])

    # Classifier head
    x = L.Dense(64, activation="relu")(x)
    x = L.Dropout(0.3)(x)
    out = L.Dense(1, activation="sigmoid")(x)  # binary probability

    model = KM.Model(inputs=[img_in, meta_in], outputs=out, name="cnn_adv_detector")
    return model

# Student/Teacher setup (for EMA teacher or FixMatch-style training)
student = build_cnn_binary()
teacher = build_cnn_binary()
teacher.set_weights(student.get_weights())  # initialize teacher = student
teacher.trainable = False                   # teacher is not updated by gradients

# Inspect model architecture
student.summary()



Model: "cnn_adv_detector"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 image (InputLayer)          [(None, 50, 50, 3)]          0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 50, 50, 32)           896       ['image[0][0]']               
                                                                                                  
 batch_normalization (Batch  (None, 50, 50, 32)           128       ['conv2d[0][0]']              
 Normalization)                                                                                   
                                                                                                  
 max_pooling2d (MaxPooling2  (None, 25, 25, 32)           0         ['batch_norma

9. Exponential Moving Average (EMA) Teacher Weight Update

In [9]:
# EMA Teacher update
def update_ema(teacher, student, alpha=EMA_ALPHA):
    """
    Update the teacher's weights as an exponential moving average (EMA) of the student's.
    Args:
        teacher: Keras model whose weights track the EMA.
        student: Keras model providing the current weights.
        alpha:   EMA decay in [0,1). Higher => slower updates (more smoothing).
                 Tip: use a lower alpha early (EMA_ALPHA_WARM) and higher later (EMA_ALPHA_LATE).
    """
    # Read current weights
    tw, sw = teacher.get_weights(), student.get_weights()

    # EMA blend: new_teacher = alpha * old_teacher + (1 - alpha) * student
    new = [alpha*t + (1.0-alpha)*s for t,s in zip(tw, sw)]

    # Write back to the teacher
    teacher.set_weights(new)

10. One Training Step: Supervised Loss + FixMatch-Style Unsupervised Consistency

In [10]:
# Training step (supervised + FixMatch unsupervised)
bce = BinaryCrossentropy(from_logits=False)     # model outputs are sigmoid probabilities
optimizer = Adam(LR)

# REDEFINE train_step (no @tf.function to avoid stale graph captures) 
def train_step(l_img, l_meta, l_y, w_img, s_img, u_meta, tau, lambda_u):
    """
    Perform a single optimization step combining:
      - Supervised loss on labeled batch: (l_img, l_meta) -> l_y
      - Unsupervised consistency loss (FixMatch):
          * Teacher predicts pseudo-labels on weak views (w_img, u_meta)
          * Student matches those labels on strong views (s_img, u_meta),
            only for samples with confidence >= tau.
    Args:
        l_img:  [B_L, H, W, 3] labeled images
        l_meta: [B_L, 1]       labeled metadata (isNight)
        l_y:    [B_L, 1]       labeled targets (0/1)
        w_img:  [B_U, H, W, 3] unlabeled weakly-augmented images
        s_img:  [B_U, H, W, 3] unlabeled strongly-augmented images
        u_meta: [B_U, 1]       unlabeled metadata (isNight)
        tau:    float          confidence threshold (e.g., get_tau(epoch))
        lambda_u: float        weight for unsupervised loss (e.g., get_lambda_u(epoch))
    Returns:
        total_loss, sup_loss, unsup_loss, mask_mean, mean_conf
    """
    with tf.GradientTape() as tape:
        # Supervised path (student)
        p_l = student([l_img, l_meta], training=True)   # predictions in [0,1]
        loss_sup = bce(l_y, p_l)

        # Pseudo-labels from teacher on weak views
        p_w = teacher([w_img, u_meta], training=False)  # shape [B_U, 1], probs
        conf = tf.maximum(p_w, 1.0 - p_w)               # confidence per sample
        mask = tf.cast(conf >= tau, tf.float32)         # 1 if confident, else 0
        y_hat = tf.cast(p_w >= 0.5, tf.float32)         # hard pseudo-labels {0,1}
        # (Optional best practice: treat pseudo-labels as constants)
        # y_hat = tf.stop_gradient(y_hat); mask = tf.stop_gradient(mask)

        # Consistency path (student on strong views)
        p_s = student([s_img, u_meta], training=True)       # shape [B_U,1]
        # elementwise BCE, masked by confidence; reduce mean over the batch
        loss_unsup = tf.reduce_mean(bce(y_hat, p_s) * mask)

        # Total loss
        loss = loss_sup + lambda_u * loss_unsup


    # Apply gradients to student only
    grads = tape.gradient(loss, student.trainable_variables)
    optimizer.apply_gradients(zip(grads, student.trainable_variables))

    # Logging helpers
    mean_conf = tf.reduce_mean(conf)
    return loss, loss_sup, loss_unsup, tf.reduce_mean(mask), mean_conf

11. Evaluation Utilities: Batched Prediction and Binary Accuracy over a Dataset

In [11]:
# Evaluation helpers
@tf.function
def predict_batch(img, meta):
    """Run a forward pass on a batch (no training)."""
    return student([img, meta], training=False)

def evaluate_dataset(ds, steps=100):
    """
    Compute simple binary accuracy over `steps` batches from a labeled dataset.
    Expects dataset elements shaped like: ((image, meta), y)
    Args:
        ds:    tf.data.Dataset yielding ((img, meta), y)
        steps: number of batches to evaluate (use len(ds) for full eval if available)
    Returns:
        acc: float accuracy in [0,1]
    """
    total, correct = 0, 0
    for i, ((img, meta), y) in enumerate(ds.take(steps)):
        # Predict probabilities and threshold at 0.5 -> {0,1}
        p = predict_batch(img, meta)
        yhat = tf.cast(p >= 0.5, tf.float32)

        # Count correct predictions in the batch
        correct += tf.reduce_sum(tf.cast(tf.equal(yhat, y), tf.int32)).numpy()
        total += y.shape[0]
    acc = correct/total if total>0 else 0.0
    return acc

12. Validation Metrics: AUROC, Macro-F1, and Expected Calibration Error (ECE)

In [12]:
def eval_val_metrics(ds, steps, model):
    """
    Evaluate a model on a labeled dataset and compute:
      - AUROC (binary)
      - Macro-F1 at 0.5 threshold
      - Expected Calibration Error (ECE) with 10 bins
    Args:
        ds:     tf.data.Dataset yielding ((img, meta), y)
        steps:  number of batches to iterate from `ds`
        model:  Keras model to evaluate (called with training=False)
    Returns:
        (auroc, f1, ece): tuple of floats
    """
    y_true, y_prob = [], []

    # Collect probabilities and ground-truth labels
    for ((img, meta), y) in ds.take(steps):
        p = model([img, meta], training=False).numpy().ravel() # predicted probs in [0,1]
        y_prob.append(p); y_true.append(y.numpy().ravel())      

    # Concatenate across all processed batches 
    y_true = np.concatenate(y_true); y_prob = np.concatenate(y_prob)

    # AUROC: guard against degenerate single-class validation windows
    auroc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true))>1 else float('nan')

    # Macro-F1 at 0.5 threshold (treat both classes equally)
    y_pred = (y_prob >= 0.5).astype(int)
    f1 = f1_score(y_true, y_pred, average="macro")

    # Expected Calibration Error (ECE) with fixed binning
    def ece_score(y_true, y_prob, bins=10):
        """Compute ECE by binning confidences and comparing bin accuracy vs. mean confidence."""
        edges = np.linspace(0,1,bins+1); e=0.0
        for i in range(bins):
            m = (y_prob>=edges[i]) & (y_prob<edges[i+1])
            if m.any():
                conf = y_prob[m].mean()                     # average confidence in bin
                acc  = (y_true[m]==(y_prob[m]>=0.5)).mean() # empirical accuracy in bin
                e += m.mean() * abs(acc-conf)               # weight by bin frequency
        return e
    
    ece = ece_score(y_true, y_prob)

    return auroc, f1, ece

13. End-to-End Training Loop: Supervised + FixMatch (Temp-Scaled Teacher, Optional Top-k), Schedulers, Metrics, Checkpoints, and CSV Logging

In [13]:
# optimizer = tf.keras.optimizers.Adam(3e-4)
bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)

# VALIDATION METRICS (AUROC/F1/ECE)
def eval_val_metrics(ds, steps, model):
    """Evaluate AUROC, macro-F1 (at 0.5), and ECE (10 bins) on a labeled dataset."""
    y_true, y_prob = [], []
    for ((img, meta), y) in ds.take(steps):
        p = model([img, meta], training=False).numpy().ravel()
        y_prob.append(p); y_true.append(y.numpy().ravel())
    y_true = np.concatenate(y_true); y_prob = np.concatenate(y_prob)

    # AUROC: guard against degenerate one-class validations
    auroc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true))>1 else float('nan')

    # Macro-F1 at 0.5
    y_pred = (y_prob >= 0.5).astype(int)
    f1 = f1_score(y_true, y_pred, average="macro")

    # Expected Calibration Error (ECE) with 10 bins
    def ece_score(y_true, y_prob, bins=10):
        edges = np.linspace(0,1,bins+1); e=0.0
        for i in range(bins):
            m = (y_prob>=edges[i]) & (y_prob<edges[i+1])
            if m.any():
                conf = y_prob[m].mean()
                acc  = (y_true[m]==(y_prob[m]>=0.5)).mean()
                e += m.mean() * abs(acc-conf)
        return e
    
    ece = ece_score(y_true, y_prob)
    return auroc, f1, ece

# Basic plan: 2-epoch warm-up (λ_u=0), then ramp-up to 1.5.
WARMUP_EPOCHS = 2
LAMBDA_U_MAX  = 1.5

def get_lambda_u(ep):
    """Unsupervised weight: 0 during warm-up, then linear ramp to LAMBDA_U_MAX by epoch 8."""
    if ep <= WARMUP_EPOCHS:
        return 0.0
    # ramp-up until the 8th epoch
    t = min(1.0, (ep - WARMUP_EPOCHS) / (8 - WARMUP_EPOCHS))
    return t * LAMBDA_U_MAX

# Threshold τ: start at 0.90, then 0.95 (can switch to adaptive below if needed)
def get_tau(ep):
    return 0.90 if ep < 2 else 0.95

# EMA α: faster tracking early, heavier smoothing later
def get_ema_alpha(ep):
    return 0.90 if ep < 3 else 0.999

# TRAIN STEP (FixMatch + Teacher Temperature + optional top-k)
@tf.function(reduce_retracing=True)
def train_step(l_img, l_meta, l_y,
               w_img, s_img, u_meta,
               tau, lambda_u,
               T=2.0,            # teacher temperature (2.0 = good starting point)
               topk_frac=0.5):   # if None: pure threshold τ; else keep top-k% confident unlabeled
    with tf.GradientTape() as tape:
        # 1) Supervised (student)
        p_l = student([l_img, l_meta], training=True)
        loss_sup = bce(l_y, p_l)

        # 2) Teacher pseudo-labels on WEAK views + Temperature scaling
        p_w = teacher([w_img, u_meta], training=False)        # [B,1] probabilities (sigmoid)
        # Convert to logits and apply temperature scaling → smoother/confidence control
        logit = tf.math.log(p_w + 1e-8) - tf.math.log(1. - p_w + 1e-8)
        p_wT = tf.sigmoid(logit / T)                          # [B,1] after temperature

        # 3) Mask: either threshold τ or top-k by confidence
        conf = tf.maximum(p_wT, 1. - p_wT)                    # binary confidence
        if topk_frac is None:
            mask = tf.cast(conf >= tau, tf.float32)
        else:
            # keep the top fraction of most-confident unlabeled samples (e.g., 50%)
            B = tf.shape(conf)[0]
            k = tf.cast(tf.math.ceil(topk_frac * tf.cast(B, tf.float32)), tf.int32)
            th = tf.sort(conf, axis=0, direction='DESCENDING')[k-1]  # confidence cutoff from top-k
            mask = tf.cast(conf >= th, tf.float32)

        y_hat = tf.cast(p_wT >= 0.5, tf.float32)              # hard pseudo-labels {0,1}

        # 4) Student on STRONG views (consistency loss, masked)
        p_s = student([s_img, u_meta], training=True)
        loss_unsup = tf.reduce_mean(bce(y_hat, p_s) * mask)

        # Total loss
        loss = loss_sup + lambda_u * loss_unsup

    # Optimize student
    grads = tape.gradient(loss, student.trainable_variables)
    optimizer.apply_gradients(zip(grads, student.trainable_variables))

    mean_conf = tf.reduce_mean(conf)
    return loss, loss_sup, loss_unsup, tf.reduce_mean(mask), mean_conf

# TRAINING LOOP (detailed; adaptive τ/λ_u; .keras checkpoint; CSV logging)
MAX_STEPS_PER_EPOCH = None             # set e.g. 300 for quicker feedback runs
PRINT_EVERY = 200

CKPT_DIR = "ckpts"; os.makedirs(CKPT_DIR, exist_ok=True)
BEST_CKPT_PATH = os.path.join(CKPT_DIR, "best_by_val_auroc.keras")  # FULL MODEL checkpoint
BEST_VAL_AUROC = -np.inf
PATIENCE = 8
no_improve_epochs = 0

CSV_LOG = "training_log.csv"
if not os.path.exists(CSV_LOG):
    with open(CSV_LOG, "w", encoding="utf-8") as f:
        f.write("epoch,tau,lambda_u,ema_alpha,used_u,val_auroc,val_f1,ece,epoch_time,val_time\n")

# Unlabeled selection policy:
# set to None for pure thresholding via τ
# set to 0.5 to keep top-50% confident (helpful if used_u ~ 100%)
TOPK_FRAC = 0.5
TEACHER_T = 2.0

steps_per_epoch = ds_steps_per_epoch if MAX_STEPS_PER_EPOCH is None else min(MAX_STEPS_PER_EPOCH, ds_steps_per_epoch)
print("Training for", EPOCHS, "epochs; steps/epoch:", steps_per_epoch)

for ep in range(1, EPOCHS+1):
    tau   = get_tau(ep)
    lam_u = get_lambda_u(ep)
    ema_a = get_ema_alpha(ep)

    u_iter = iter(ds_unlabeled)
    l_iter = iter(ds_train_l)

    sup_losses, unsup_losses, masks, confs = [], [], [], []

    t_ep0 = time.time()
    # Tiny warmup step (stabilizes graphs/caches)
    try:
        (l_img, l_meta), l_y = next(l_iter)
        w_img, s_img, u_meta = next(u_iter)
        _ = train_step(l_img, l_meta, l_y, w_img, s_img, u_meta,
                       tau=tau, lambda_u=lam_u, T=TEACHER_T, topk_frac=TOPK_FRAC)
    except StopIteration:
        pass

    # main epoch loop 
    for step in range(1, steps_per_epoch+1):
        try:
            (l_img, l_meta), l_y = next(l_iter)
        except StopIteration:
            l_iter = iter(ds_train_l)
            (l_img, l_meta), l_y = next(l_iter)

        try:
            w_img, s_img, u_meta = next(u_iter)
        except StopIteration:
            u_iter = iter(ds_unlabeled)
            w_img, s_img, u_meta = next(u_iter)

        loss, l_sup, l_unsup, m, mean_conf = train_step(
            l_img, l_meta, l_y, w_img, s_img, u_meta,
            tau=tau, lambda_u=lam_u, T=TEACHER_T, topk_frac=TOPK_FRAC
        )

        sup_losses.append(float(l_sup.numpy()))
        unsup_losses.append(float(l_unsup.numpy()))
        masks.append(float(m.numpy()))
        confs.append(float(mean_conf.numpy()))

        # EMA update for the teacher
        update_ema(teacher, student, alpha=ema_a)

        # Mini-log
        if step % PRINT_EVERY == 0 or step == 1:
            elapsed = time.time() - t_ep0
            sps = step / max(1e-6, elapsed)
            used_u_pct = 100.0 * (np.mean(masks) if masks else 0.0)
            print(f"  ep{ep:02d} step{step:05d}/{steps_per_epoch} "
                  f"| sps={sps:.2f} "
                  f"| sup={np.mean(sup_losses):.4f} unsup={np.mean(unsup_losses):.4f} "
                  f"| used_u={used_u_pct:.1f}% conf={np.mean(confs):.3f}")

    # validation 
    t_val0 = time.time()
    auroc, f1, ece = eval_val_metrics(ds_val_l, val_steps, student)
    t_val = time.time() - t_val0

    t_epoch = time.time() - t_ep0
    used_u_pct = 100.0 * (np.mean(masks) if masks else 0.0)

    print(f"[Epoch {ep:02d}] tau={tau:.2f} lu={lam_u:.2f} ema={ema_a:.3f} "
          f"| sup={np.mean(sup_losses):.4f} unsup={np.mean(unsup_losses):.4f} "
          f"| used_u={used_u_pct:.1f}% avg_conf={np.mean(confs):.3f} "
          f"| VAL: AUROC={auroc:.3f} F1={f1:.3f} ECE={ece:.3f} "
          f"| epoch_time={t_epoch:.1f}s val_time={t_val:.1f}s")

    # Append to CSV log
    with open(CSV_LOG, "a", encoding="utf-8") as f:
        f.write(f"{ep},{tau:.4f},{lam_u:.4f},{ema_a:.4f},{used_u_pct:.2f},{auroc:.6f},{f1:.6f},{ece:.6f},{t_epoch:.3f},{t_val:.3f}\n")

    # Checkpoints (.keras full model)
    if np.isfinite(auroc) and auroc > BEST_VAL_AUROC:
        BEST_VAL_AUROC = auroc
        student.save(BEST_CKPT_PATH)
        no_improve_epochs = 0
        print(f"  ✓ Saved best (val AUROC={BEST_VAL_AUROC:.3f}) → {BEST_CKPT_PATH}")
    else:
        no_improve_epochs += 1
        print(f"  ↳ No AUROC improvement ({no_improve_epochs}/{PATIENCE})")

    # Simple adaptive control for next epoch 
    # If too many unlabeled get accepted, tighten things a bit
    if used_u_pct >= 95.0:
        TEACHER_T = min(3.0, TEACHER_T + 0.25)   # softer teacher → lower confidence
        LAMBDA_U_MAX = max(0.8, LAMBDA_U_MAX * 0.9)
        TOPK_FRAC = max(0.4, TOPK_FRAC - 0.05)   # e.g., 0.50 → 0.45

    # If too few (<20%) get accepted, relax slightly
    if used_u_pct < 20.0:
        TEACHER_T = max(1.25, TEACHER_T - 0.25)
        LAMBDA_U_MAX = min(1.5, LAMBDA_U_MAX * 1.05)
        TOPK_FRAC = min(0.7, TOPK_FRAC + 0.05)

    # Early stopping on val AUROC
    if no_improve_epochs >= PATIENCE:
        print("Early stopping: patience reached.")
        break

# final full save
FINAL_MODEL_PATH = "../../models/model_v_semi.keras"
os.makedirs(os.path.dirname(FINAL_MODEL_PATH), exist_ok=True)
student.save(FINAL_MODEL_PATH)
print("Saved final:", FINAL_MODEL_PATH)

Training for 30 epochs; steps/epoch: 1467
PATH BEFORE: "../../datasets/adv_outputs/ANN_fgsm/processed_images/train/img_9695.png"
PATH BEFORE: "../../datasets/adv_outputs/CNN_fgsm/processed_images/train/img_15538.png"
PATH BEFORE: "../../datasets/processed_images/train/img_22231.jpg"
PATH BEFORE: "../../datasets/processed_images/train/img_9840.jpg"
PATH BEFORE: "../../datasets/adv_outputs/CNN_fgsm/processed_images/train/img_5532.png"
PATH BEFORE: "../../datasets/adv_outputs/ANN_fgsm/processed_images/train/img_11525.png"
PATH BEFORE: "../../datasets/processed_images/train/img_13990.jpg"
PATH BEFORE: "../../datasets/adv_outputs/CNN_fgsm/processed_images/train/img_6071.png"
PATH BEFORE: "../../datasets/adv_outputs/ANN_fgsm/processed_images/train/img_24086.png"
PATH BEFORE: "../../datasets/processed_images/train/img_27158.jpg"
PATH BEFORE: "../../datasets/adv_outputs/CNN_fgsm/processed_images/train/img_2935.png"
PATH BEFORE: "../../datasets/adv_outputs/ANN_fgsm/processed_images/train/img_18

NotFoundError: {{function_node __wrapped__IteratorGetNext_output_types_3_device_/job:localhost/replica:0/task:0/device:CPU:0}} NewRandomAccessFile failed to Create/Open: ../../datasets/processed_images/train/img_30944.jpg : The system cannot find the file specified.
; No such file or directory
	 [[{{node ReadFile}}]] [Op:IteratorGetNext] name: 

14. Test-Time Evaluation: Load Model, Compute Metrics (Report, AUROC)

In [ ]:
# CONFIG 
MODEL_PATH = "../../models/model_v_semi.keras"   

# DATASET HANDLES 
ds_eval = ds_test_l
steps = test_steps

# LOAD MODEL
model = tf.keras.models.load_model(MODEL_PATH)
print("Loaded model:", MODEL_PATH)

# COLLECT PREDICTIONS 
y_true, y_prob = [], []
for ((img, meta), y) in ds_eval.take(steps):
    p = model([img, meta], training=False).numpy().ravel()
    y_prob.append(p)
    y_true.append(y.numpy().ravel())

y_true = np.concatenate(y_true)
y_prob  = np.concatenate(y_prob)
# Threshold at 0.5 (if you have a calibrated decision threshold, use that instead)
y_pred  = (y_prob >= 0.5).astype(int)    

# METRICS 
# Classification report (per-class precision/recall/f1/support + macro/weighted)
target_names = ["clean(0)", "adv(1)"]   # προσαρμογή ονομάτων
report = classification_report(y_true, y_pred, target_names=target_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
print("\nClassification report (summary):\n", report_df[["precision","recall","f1-score","support"]])


# AUROC (optional but useful for probabilistic quality)
try:
    auroc = roc_auc_score(y_true, y_prob)
    print(f"\nAUROC: {auroc:.6f}")
except Exception as e:
    print("AUROC not computed:", e)

# CONFUSION MATRIX 
cm = confusion_matrix(y_true, y_pred, labels=[0,1])
cm_norm = confusion_matrix(y_true, y_pred, labels=[0,1], normalize="true")

print("\nConfusion matrix (abs):\n", cm)
print("\nConfusion matrix (normalized by true):\n", cm_norm)

Loaded model: models/model_v_semi.keras

Classification report (summary):
               precision    recall  f1-score      support
clean(0)       0.999569  0.999741  0.999655  11599.00000
adv(1)         0.999870  0.999784  0.999827  23153.00000
accuracy       0.999770  0.999770  0.999770      0.99977
macro avg      0.999720  0.999763  0.999741  34752.00000
weighted avg   0.999770  0.999770  0.999770  34752.00000

AUROC: 0.999999

Confusion matrix (abs):
 [[11596     3]
 [    5 23148]]

Confusion matrix (normalized by true):
 [[9.99741357e-01 2.58642986e-04]
 [2.15954736e-04 9.99784045e-01]]


' fig, ax = plt.subplots(figsize=(5,4), dpi=130)\nim = ax.imshow(cm, interpolation=\'nearest\')\nax.set_title("Confusion Matrix (absolute)")\nax.set_xticks([0,1]); ax.set_yticks([0,1])\nax.set_xticklabels(target_names); ax.set_yticklabels(target_names)\nplt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")\nfor i in range(cm.shape[0]):\n    for j in range(cm.shape[1]):\n        ax.text(j, i, cm[i, j], ha="center", va="center")\nax.set_ylabel("True label"); ax.set_xlabel("Predicted label")\nfig.tight_layout()\nfig.savefig(CONF_MAT_PNG, bbox_inches="tight")\nplt.close(fig)\n\n# Plot (normalized)\nfig, ax = plt.subplots(figsize=(5,4), dpi=130)\nim = ax.imshow(cm_norm, interpolation=\'nearest\')\nax.set_title("Confusion Matrix (normalized)")\nax.set_xticks([0,1]); ax.set_yticks([0,1])\nax.set_xticklabels(target_names); ax.set_yticklabels(target_names)\nplt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")\nfor i in range(cm_norm.shape[0])